# Galleria Intelligence Platform

## Clean Data Science Capstone Notebook

This notebook turns verified receipt data into a reusable retail intelligence pipeline:

```text
Load → Understand → Clean → Extract → Evaluate → Analyse → Recommend → Forecast → Export
```

**Current capstone workbook:** 82 receipt IDs, 96 transaction lines, and 79 purchasing customer IDs. The small 45-receipt public demo in this repository is an application fixture, not the capstone benchmark.

**Important:** the OCR models are pretrained. Verified receipts provide ground truth for evaluation; they are not used to train a new vision model.

Customer names, phone numbers, receipt images, and raw OCR predictions are saved only to protected private storage.

## Capstone modelling update

The current workbook scope is **82 receipt IDs, 96 transaction lines, and 79 purchasing customer IDs**. Of the 79 customers, 77 have a single recorded receipt; RFM frequency is therefore descriptive only. Dates occur in two periods separated by 201 days. Weekly GMV modelling is limited to the latest continuous weekly period and is an exploratory next-week estimate, never a seasonal or long-range forecast.

Use the canonical receipt table to preserve missing-date and missing-total flags, then exclude those rows from modelling without imputing values. Purchase-pattern K-Means groups complete receipts (not customers) using standardized GMV, units, line count, product/alteration mix, payment state, and outstanding balance. Consider 2–6 groups and select the highest silhouette score.

For OCR, keep Qwen 2.5-VL as the primary document-to-JSON extractor and benchmark PaddleOCR PP-StructureV3 as a second pretrained structured-document pipeline. Canonicalize image names before joining predictions to ground truth. With all images available, use a fixed 60-development / 22-held-out receipt split. Do not tune against the held-out set and do not claim fine-tuning. Export only de-identified metrics: JSON and missing-field rates, normalized exact match, BHD ±0.001 numeric accuracy, line-item precision/recall/F1, median processing time, and error types.

## Run this notebook in Google Colab

1. In Google Drive, keep the files exactly as shown: `My Drive/NEW_GALLERIA/Galleria Receipt Data.xlsx` and `My Drive/NEW_GALLERIA/receipts/`.
2. Upload this notebook to Colab, then choose **Runtime → Run all** and approve the Google Drive mount.
3. Leave `RUN_OCR = False` for the first run. It will create `NEW_GALLERIA/outputs/public/` without making OCR calls.
4. Add `HF_TOKEN` to Colab Secrets only when ready to run the Qwen OCR evaluation. Start with `OCR_LIMIT = 5`.
5. Configure only on the fixed 60-receipt development split. Run the final 22-receipt holdout evaluation once; do not tune against its results.

The OCR cache prevents successfully processed receipts from being charged twice. Names, phones, raw predictions, and original images remain in private Drive storage.

## 1. Install

In [ ]:
%pip -q install -U huggingface-hub openpyxl seaborn matplotlib

## 2. Imports and setup

This section imports the required libraries, connects Google Drive, and defines the project folders.

In [ ]:
import base64
import json
import os
import re
import time
import warnings
from datetime import datetime, timezone
from difflib import SequenceMatcher
from io import BytesIO
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image, ImageOps

try:
    from IPython.display import JSON, display
except ImportError:
    def display(value):
        print(value)

    def JSON(value):
        return value

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)
RANDOM_STATE = 42

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    print("Google Drive connected.")
except ImportError:
    print("Not running in Colab. Using the local project path if supplied.")

In [ ]:
# Drive layout expected by this Colab notebook:
# My Drive/NEW_GALLERIA/Galleria Receipt Data.xlsx
# My Drive/NEW_GALLERIA/receipts/
DEFAULT_PROJECT = "/content/drive/MyDrive/NEW_GALLERIA"
PROJECT_DIR = Path(os.getenv("GALLERIA_PROJECT_DIR", DEFAULT_PROJECT))
WORKBOOK_NAME = "Galleria Receipt Data.xlsx"
WORKBOOK_PATH = PROJECT_DIR / WORKBOOK_NAME
RECEIPT_DIR = PROJECT_DIR / "receipts"
OUTPUT_DIR = PROJECT_DIR / "outputs"
PUBLIC_DIR = OUTPUT_DIR / "public"
PRIVATE_DIR = OUTPUT_DIR / "private"
FIGURE_DIR = OUTPUT_DIR / "figures"

HF_MODEL = "Qwen/Qwen2.5-VL-3B-Instruct"
HF_PROVIDER = "featherless-ai"
RUN_OCR = False  # Inspect and export data before running paid OCR calls.
OCR_LIMIT = 5
FORCE_REPROCESS = False

if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        f"Could not find {PROJECT_DIR}. In Google Drive, create My Drive/NEW_GALLERIA and add the workbook plus receipts folder."
    )
for folder in [PUBLIC_DIR, PRIVATE_DIR, FIGURE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)
print("Workbook expected:", WORKBOOK_PATH)
print("Receipt images folder:", RECEIPT_DIR)

## 3. Load Excel data and receipt images

The workbook uses three instruction rows above the real column headers, so `header=3` is required.

In [ ]:
if not WORKBOOK_PATH.exists():
    candidates = sorted(path for path in PROJECT_DIR.glob("*.xlsx") if not path.name.startswith("~$"))
    if len(candidates) == 1:
        WORKBOOK_PATH = candidates[0]
        print("Using detected workbook:", WORKBOOK_PATH.name)
    else:
        raise FileNotFoundError(
            f"Could not find {WORKBOOK_NAME}. Found {len(candidates)} usable Excel files in {PROJECT_DIR}."
        )

transactions_raw = pd.read_excel(WORKBOOK_PATH, sheet_name="Transactions", header=3)
customers_raw = pd.read_excel(
    WORKBOOK_PATH, sheet_name="Customers_Private", header=3, dtype={"phone_number": str}
)

print("Workbook used:", WORKBOOK_PATH.name)
print("Transaction rows loaded:", len(transactions_raw))
print("Customer-register rows loaded:", len(customers_raw))

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}

def canonical_image_key(value):
    """Make receipt-1.jpg and receipt-01.JPG match."""
    if pd.isna(value):
        return None
    stem = Path(str(value).strip()).stem.lower()
    numbers = re.findall(r"\d+", stem)
    if numbers:
        return f"receipt-{int(numbers[-1])}"
    return re.sub(r"[^a-z0-9]+", "-", stem).strip("-")

def image_sort_value(path):
    numbers = re.findall(r"\d+", Path(path).stem)
    return int(numbers[-1]) if numbers else 10**9

receipt_files = sorted(
    [p for p in RECEIPT_DIR.rglob("*") if p.suffix.lower() in IMAGE_EXTENSIONS],
    key=image_sort_value,
) if RECEIPT_DIR.exists() else []

image_index = {canonical_image_key(path.name): path for path in receipt_files}
print("Receipt images found:", len(receipt_files))

In [ ]:
display(transactions_raw.head(8))
display(customers_raw.head(8))

## 4. Understand the dataset

Before cleaning or modelling, confirm the available records, missing fields, date coverage, and brand coverage.

In [ ]:
REQUIRED_COLUMNS = {
    "image_name", "receipt_id", "line_number", "transaction_date",
    "customer_id", "transaction_type", "brand_clean",
    "description_raw", "product_category", "quantity",
    "unit_price_bhd", "receipt_total_bhd", "payment_status",
}

transactions_raw.columns = transactions_raw.columns.astype(str).str.strip()
customers_raw.columns = customers_raw.columns.astype(str).str.strip()

missing_columns = REQUIRED_COLUMNS - set(transactions_raw.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

transactions = transactions_raw[
    transactions_raw["image_name"].notna()
    & transactions_raw["receipt_id"].notna()
].copy()

used_customer_ids = set(transactions["customer_id"].dropna().astype(str))
customers = customers_raw[
    customers_raw["customer_id"].astype(str).isin(used_customer_ids)
].copy()

In [ ]:
raw_dates = pd.to_datetime(transactions["transaction_date"], errors="coerce")
brand_known = ~transactions["brand_clean"].fillna("Unknown").eq("Unknown")

dataset_overview = pd.DataFrame({
    "Measure": [
        "Unique receipts", "Item/service rows", "Customer IDs used",
        "Multi-line receipts", "Confirmed-brand rows",
        "Missing dates", "Receipt images found",
    ],
    "Value": [
        transactions["receipt_id"].nunique(), len(transactions),
        transactions["customer_id"].nunique(),
        int((transactions.groupby("receipt_id").size() > 1).sum()),
        int(brand_known.sum()), int(raw_dates.isna().sum()),
        len(receipt_files),
    ],
})

display(dataset_overview)
print("Date range:", raw_dates.min(), "to", raw_dates.max())

In [ ]:
missingness = (
    transactions.isna().mean().mul(100)
    .sort_values(ascending=False)
    .rename("missing_percent")
    .rename_axis("field")
    .reset_index()
)

display(missingness.head(15))

plt.figure(figsize=(10, 5))
sns.barplot(data=missingness.head(15), x="missing_percent", y="field", color="#8A6A3B")
plt.title("Most incomplete transaction fields")
plt.xlabel("Missing values (%)")
plt.ylabel("")
plt.tight_layout()
plt.show()

## 5. Clean and validate transactions

Cleaning creates consistent dates and numbers, preserves missing values, prevents multi-item receipts from double-counting sales, and derives the current payment position.

In [ ]:
DATE_COLUMNS = ["transaction_date", "balance_paid_date"]
NUMERIC_COLUMNS = [
    "quantity", "unit_price_bhd", "line_total_bhd",
    "receipt_total_bhd", "advance_paid_bhd", "balance_bhd",
]

transactions["image_key"] = transactions["image_name"].map(canonical_image_key)
transactions["receipt_id"] = (
    transactions["receipt_id"].astype(str)
    .str.replace(r"\.0$", "", regex=True).str.strip()
)
transactions["customer_id"] = transactions["customer_id"].astype(str).str.strip()
transactions["line_number"] = (
    pd.to_numeric(transactions["line_number"], errors="coerce")
    .fillna(1).astype(int)
)

for column in DATE_COLUMNS:
    transactions[column] = pd.to_datetime(transactions[column], errors="coerce")
for column in NUMERIC_COLUMNS:
    transactions[column] = pd.to_numeric(transactions[column], errors="coerce")

In [ ]:
missing_line_input = (
    transactions["quantity"].isna()
    | transactions["unit_price_bhd"].isna()
)

transactions["line_total_clean"] = transactions["line_total_bhd"].mask(
    missing_line_input
)
calculated_line_total = (
    transactions["quantity"] * transactions["unit_price_bhd"]
)
transactions["line_total_clean"] = (
    transactions["line_total_clean"].fillna(calculated_line_total)
)

duplicate_lines = transactions.duplicated(
    ["receipt_id", "line_number"], keep=False
)
print("Duplicate receipt-line combinations:", int(duplicate_lines.sum()))
display(transactions.head(8))

In [ ]:
ordered = transactions.sort_values(["receipt_id", "line_number"])
receipts = ordered.drop_duplicates("receipt_id").copy()

item_summary = ordered.groupby("receipt_id", as_index=False).agg(
    item_rows=("line_number", "size"),
    units=("quantity", lambda values: values.sum(min_count=1)),
    item_total_sum=("line_total_clean", lambda values: values.sum(min_count=1)),
    known_item_totals=("line_total_clean", "count"),
)

receipts = receipts.merge(item_summary, on="receipt_id", how="left")
all_lines_known = receipts["known_item_totals"].eq(receipts["item_rows"])
receipts["receipt_total_clean"] = receipts["receipt_total_bhd"]
can_derive = receipts["receipt_total_clean"].isna() & all_lines_known
receipts.loc[can_derive, "receipt_total_clean"] = receipts.loc[can_derive, "item_total_sum"]
receipts["receipt_total_source"] = np.select(
    [receipts["receipt_total_bhd"].notna(), can_derive],
    ["Receipt", "Derived from complete item lines"],
    default="Missing",
)

In [ ]:
status = receipts["payment_status"].fillna("").str.strip().str.lower()
paid_now = status.eq("paid") | receipts["balance_paid_date"].notna()
open_balance = status.isin(["partially paid", "unpaid", "balance due"]) & ~paid_now

receipts["current_outstanding_bhd"] = np.nan
receipts.loc[paid_now, "current_outstanding_bhd"] = 0.0
receipts.loc[open_balance, "current_outstanding_bhd"] = receipts.loc[
    open_balance, "balance_bhd"
]

calculated_balance = (
    receipts["receipt_total_clean"]
    - receipts["advance_paid_bhd"].fillna(0)
)
needs_balance = open_balance & receipts["current_outstanding_bhd"].isna()
receipts.loc[needs_balance, "current_outstanding_bhd"] = calculated_balance[needs_balance]
receipts["current_outstanding_bhd"] = receipts["current_outstanding_bhd"].clip(lower=0)
receipts["amount_paid_bhd"] = receipts["receipt_total_clean"] - receipts["current_outstanding_bhd"]

In [ ]:
paid_formula_placeholder = (
    status.eq("paid")
    & receipts["advance_paid_bhd"].fillna(0).eq(0)
    & receipts["receipt_total_bhd"].notna()
    & receipts["balance_bhd"].eq(receipts["receipt_total_bhd"])
)

receipts["advance_truth_bhd"] = receipts["advance_paid_bhd"].mask(
    paid_formula_placeholder
)
receipts["balance_truth_bhd"] = receipts["balance_bhd"].mask(
    paid_formula_placeholder
)

receipts["final_payment_status"] = np.select(
    [
        receipts["current_outstanding_bhd"].eq(0),
        receipts["current_outstanding_bhd"].gt(0) & receipts["amount_paid_bhd"].gt(0),
        receipts["current_outstanding_bhd"].gt(0),
    ],
    ["Paid", "Partially Paid", "Unpaid"],
    default="Unknown",
)

In [ ]:
missing_images = sorted(set(receipts["image_key"]) - set(image_index))
extra_images = sorted(set(image_index) - set(receipts["image_key"]))
line_total_mismatch = (
    receipts["receipt_total_clean"].notna()
    & receipts["item_total_sum"].notna()
    & (receipts["receipt_total_clean"] - receipts["item_total_sum"]).abs().gt(0.01)
)

quality_report = pd.DataFrame({
    "Check": [
        "Duplicate receipt-line rows", "Missing transaction dates",
        "Missing receipt totals", "Unknown brand item rows",
        "Receipt total vs item total mismatches",
        "Receipt images not found", "Images without verified labels",
    ],
    "Count": [
        int(duplicate_lines.sum()), int(receipts["transaction_date"].isna().sum()),
        int(receipts["receipt_total_clean"].isna().sum()),
        int(transactions["brand_clean"].fillna("Unknown").eq("Unknown").sum()),
        int(line_total_mismatch.sum()), len(missing_images), len(extra_images),
    ],
})

display(quality_report)

## 6. OCR receipt extraction

The notebook uses a pretrained vision-language model to convert each receipt image into structured JSON. The model is evaluated later against the verified workbook.

Model: `Qwen/Qwen2.5-VL-3B-Instruct` via `featherless-ai`

References:

- https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct
- https://huggingface.co/docs/huggingface_hub/package_reference/inference_client

In [ ]:
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except (ImportError, KeyError, TypeError):
    HF_TOKEN = os.getenv("HF_TOKEN")

try:
    from huggingface_hub import InferenceClient
except ImportError:
    InferenceClient = None

client = None
if HF_TOKEN and InferenceClient:
    client = InferenceClient(provider=HF_PROVIDER, api_key=HF_TOKEN)

print("HF token available:", bool(HF_TOKEN))
print("OCR model:", HF_MODEL)

In [ ]:
OCR_SCHEMA = {
    "receipt_id": None,
    "transaction_date": None,
    "customer_name": None,
    "phone_number": None,
    "items": [{
        "description_raw": None,
        "brand_raw": None,
        "product_category": None,
        "color": None,
        "quantity": None,
        "unit_price_bhd": None,
        "line_total_bhd": None,
        "alteration_required": None,
        "alteration_details": None,
    }],
    "receipt_total_bhd": None,
    "advance_paid_bhd": None,
    "balance_bhd": None,
    "payment_status": None,
    "balance_paid_date": None,
}

OCR_PROMPT = f"""Extract this handwritten Galleria receipt.
Return one valid JSON object only, using this schema:
{json.dumps(OCR_SCHEMA, indent=2)}
Use YYYY-MM-DD dates and JSON numbers for BHD amounts.
Keep separate receipt lines as separate items in top-to-bottom order.
Never guess an unreadable value or brand; use null.
payment_status must be Paid, Partially Paid, Unpaid, or null."""

In [ ]:
def image_to_data_url(path, max_size=1800, quality=88):
    image = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
    image.thumbnail((max_size, max_size))
    buffer = BytesIO()
    image.save(buffer, format="JPEG", quality=quality, optimize=True)
    encoded = base64.b64encode(buffer.getvalue()).decode("ascii")
    return f"data:image/jpeg;base64,{encoded}"

def parse_json_object(text):
    cleaned = re.sub(
        r"^```(?:json)?\s*|\s*```$", "", text.strip(),
        flags=re.IGNORECASE | re.DOTALL,
    )
    start = cleaned.find("{")
    if start < 0:
        raise ValueError("The model did not return a JSON object.")
    result, _ = json.JSONDecoder().raw_decode(cleaned[start:])
    return result

In [ ]:
def extract_receipt(image_path):
    if client is None:
        raise ValueError("Add HF_TOKEN to Colab Secrets before running OCR.")

    started = time.perf_counter()
    response = client.chat.completions.create(
        model=HF_MODEL,
        messages=[{
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": image_to_data_url(image_path)}},
                {"type": "text", "text": OCR_PROMPT},
            ],
        }],
        max_tokens=1400,
        temperature=0.0,
    )

    prediction = parse_json_object(response.choices[0].message.content)
    return {
        "image_name": image_path.name,
        "image_key": canonical_image_key(image_path.name),
        "processing_seconds": round(time.perf_counter() - started, 3),
        "prediction": prediction,
        "error": None,
    }

In [ ]:
OCR_CACHE_PATH = PRIVATE_DIR / "ocr_predictions_private.json"

if OCR_CACHE_PATH.exists():
    ocr_predictions = json.loads(OCR_CACHE_PATH.read_text(encoding="utf-8"))
else:
    ocr_predictions = []

cached_by_key = {record["image_key"]: record for record in ocr_predictions}
selected_images = receipt_files if OCR_LIMIT is None else receipt_files[:OCR_LIMIT]

if RUN_OCR:
    for number, image_path in enumerate(selected_images, 1):
        key = canonical_image_key(image_path.name)
        if key in cached_by_key and not FORCE_REPROCESS:
            print(f"{number}/{len(selected_images)} cached: {image_path.name}")
            continue
        try:
            cached_by_key[key] = extract_receipt(image_path)
            print(f"{number}/{len(selected_images)} extracted: {image_path.name}")
        except Exception as error:
            cached_by_key[key] = {
                "image_name": image_path.name, "image_key": key,
                "prediction": None, "processing_seconds": None,
                "error": str(error),
            }
            print(f"{number}/{len(selected_images)} failed: {error}")
        OCR_CACHE_PATH.write_text(
            json.dumps(list(cached_by_key.values()), indent=2), encoding="utf-8"
        )

ocr_predictions = list(cached_by_key.values())

In [ ]:
successful_ocr = [
    record for record in ocr_predictions
    if record.get("prediction") and not record.get("error")
]
failed_ocr = [record for record in ocr_predictions if record.get("error")]

print("Successful OCR extractions:", len(successful_ocr))
print("Failed OCR extractions:", len(failed_ocr))

if successful_ocr:
    sample = successful_ocr[0]
    if sample["image_key"] in image_index:
        display(Image.open(image_index[sample["image_key"]]))
    display(JSON(sample["prediction"]))
else:
    print("No predictions yet. Set RUN_OCR = True and run this section.")

## 7. OCR testing and evaluation

The verified workbook is the ground truth. Missing or unknown manual values are excluded rather than counted as correct or incorrect.

Accuracy is reported by field, with error examples and complete-receipt accuracy. Customer-name and phone evaluation remain private.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
all_keys = np.array(sorted(receipts["image_key"].dropna().unique()))
DEVELOPMENT_RECEIPTS = 60
HOLDOUT_RECEIPTS = 22
expected_receipts = DEVELOPMENT_RECEIPTS + HOLDOUT_RECEIPTS
if len(all_keys) != expected_receipts:
    raise ValueError(
        f"Expected {expected_receipts} labelled receipt IDs for the fixed OCR split; found {len(all_keys)}. Resolve missing or duplicate image keys before evaluation."
    )
holdout_keys = set(rng.choice(all_keys, size=HOLDOUT_RECEIPTS, replace=False))

receipts["evaluation_split"] = np.where(
    receipts["image_key"].isin(holdout_keys), "Holdout test", "Development"
)

print(receipts["evaluation_split"].value_counts())
print("Do not change the OCR prompt after reviewing holdout-test errors.")

In [ ]:
UNKNOWN_VALUES = {"", "unknown", "not stated", "n/a", "none", "null", "nan"}
NUMERIC_FIELDS = {
    "quantity", "unit_price_bhd", "line_total_bhd",
    "receipt_total_bhd", "advance_paid_bhd", "balance_bhd",
}
DATE_FIELDS = {"transaction_date", "balance_paid_date"}
PHONE_FIELDS = {"phone_number"}

def normalize_text(value):
    if pd.isna(value):
        return ""
    cleaned = re.sub(r"\s+", " ", str(value).strip().lower())
    return "" if cleaned in UNKNOWN_VALUES else cleaned

def normalize_number(value):
    try:
        number = float(value)
        return number if np.isfinite(number) else np.nan
    except (TypeError, ValueError):
        return np.nan

def normalize_date(value):
    parsed = pd.to_datetime(value, errors="coerce", dayfirst=True)
    return "" if pd.isna(parsed) else parsed.strftime("%Y-%m-%d")

def normalize_phone(value):
    if pd.isna(value):
        return ""
    digits = re.sub(r"\D", "", str(value).replace("'", ""))
    return digits if len(digits) >= 8 else ""

In [ ]:
def compare_value(actual, predicted, field):
    if field in NUMERIC_FIELDS:
        actual_norm = normalize_number(actual)
        predicted_norm = normalize_number(predicted)
        evaluated = not pd.isna(actual_norm)
        correct = evaluated and not pd.isna(predicted_norm) and abs(actual_norm - predicted_norm) <= 0.01
        similarity = 100.0 if correct else (0.0 if evaluated else np.nan)
    elif field in DATE_FIELDS:
        actual_norm = normalize_date(actual)
        predicted_norm = normalize_date(predicted)
        evaluated = bool(actual_norm)
        correct = evaluated and actual_norm == predicted_norm
        similarity = 100.0 if correct else (0.0 if evaluated else np.nan)
    elif field in PHONE_FIELDS:
        actual_norm = normalize_phone(actual)
        predicted_norm = normalize_phone(predicted)
        evaluated = bool(actual_norm)
        correct = evaluated and actual_norm == predicted_norm
        similarity = 100.0 if correct else (0.0 if evaluated else np.nan)
    else:
        actual_norm = normalize_text(actual)
        predicted_norm = normalize_text(predicted)
        evaluated = bool(actual_norm)
        correct = evaluated and actual_norm == predicted_norm
        similarity = (
            SequenceMatcher(None, actual_norm, predicted_norm).ratio() * 100
            if evaluated else np.nan
        )

    return {
        "evaluated": evaluated,
        "is_correct": int(correct) if evaluated else np.nan,
        "text_similarity": similarity,
    }

In [ ]:
customer_lookup = customers.set_index("customer_id").to_dict("index")
receipt_lookup = receipts.set_index("image_key")
evaluation_rows = []

RECEIPT_FIELDS = {
    "receipt_id": "receipt_id",
    "transaction_date": "transaction_date",
    "receipt_total_bhd": "receipt_total_clean",
    "advance_paid_bhd": "advance_truth_bhd",
    "balance_bhd": "balance_truth_bhd",
    "payment_status": "final_payment_status",
}

for record in successful_ocr:
    key = record["image_key"]
    if key not in receipt_lookup.index:
        continue
    truth = receipt_lookup.loc[key]
    prediction = record["prediction"]
    for predicted_field, truth_field in RECEIPT_FIELDS.items():
        result = compare_value(truth.get(truth_field), prediction.get(predicted_field), predicted_field)
        evaluation_rows.append({
            "image_key": key, "split": truth["evaluation_split"],
            "level": "receipt", "field": predicted_field,
            "actual": truth.get(truth_field), "predicted": prediction.get(predicted_field),
            "processing_seconds": record.get("processing_seconds"), **result,
        })

In [ ]:
for record in successful_ocr:
    key = record["image_key"]
    if key not in receipt_lookup.index:
        continue
    truth = receipt_lookup.loc[key]
    prediction = record["prediction"]
    private_customer = customer_lookup.get(truth.get("customer_id"), {})

    for field in ["customer_name", "phone_number"]:
        result = compare_value(private_customer.get(field), prediction.get(field), field)
        evaluation_rows.append({
            "image_key": key, "split": truth["evaluation_split"],
            "level": "private", "field": field,
            "actual": private_customer.get(field), "predicted": prediction.get(field),
            "processing_seconds": record.get("processing_seconds"), **result,
        })

In [ ]:
ITEM_FIELDS = {
    "description_raw": ("description_raw", "description_raw"),
    "brand_raw": ("brand_clean", "brand"),
    "product_category": ("product_category", "product_category"),
    "color": ("color", "color"),
    "quantity": ("quantity", "quantity"),
    "unit_price_bhd": ("unit_price_bhd", "unit_price_bhd"),
    "line_total_bhd": ("line_total_clean", "line_total_bhd"),
    "alteration_required": ("alteration_required", "alteration_required"),
}

for record in successful_ocr:
    key = record["image_key"]
    if key not in receipt_lookup.index:
        continue
    split = receipt_lookup.loc[key, "evaluation_split"]
    true_items = transactions[transactions["image_key"].eq(key)].sort_values("line_number")
    predicted_items = record["prediction"].get("items") or []

    count_result = compare_value(len(true_items), len(predicted_items), "quantity")
    evaluation_rows.append({
        "image_key": key, "split": split, "level": "receipt",
        "field": "item_count", "actual": len(true_items),
        "predicted": len(predicted_items),
        "processing_seconds": record.get("processing_seconds"), **count_result,
    })

In [ ]:
for record in successful_ocr:
    key = record["image_key"]
    if key not in receipt_lookup.index:
        continue
    split = receipt_lookup.loc[key, "evaluation_split"]
    true_items = transactions[transactions["image_key"].eq(key)].sort_values("line_number")
    predicted_items = record["prediction"].get("items") or []

    for position, (_, true_item) in enumerate(true_items.iterrows()):
        predicted_item = predicted_items[position] if position < len(predicted_items) else {}
        predicted_item = predicted_item if isinstance(predicted_item, dict) else {}
        for predicted_field, (truth_field, metric_field) in ITEM_FIELDS.items():
            result = compare_value(true_item.get(truth_field), predicted_item.get(predicted_field), metric_field)
            evaluation_rows.append({
                "image_key": key, "split": split, "level": "item",
                "field": metric_field, "actual": true_item.get(truth_field),
                "predicted": predicted_item.get(predicted_field),
                "processing_seconds": record.get("processing_seconds"), **result,
            })

In [ ]:
ocr_evaluation = pd.DataFrame(evaluation_rows)

if not ocr_evaluation.empty:
    evaluated = ocr_evaluation[ocr_evaluation["evaluated"]].copy()
    ocr_metrics = evaluated.groupby(
        ["split", "level", "field"], as_index=False
    ).agg(
        evaluated_values=("is_correct", "size"),
        exact_accuracy=("is_correct", "mean"),
        average_similarity=("text_similarity", "mean"),
    )

    accuracy_by_split = evaluated.groupby("split")["is_correct"].mean()
    receipt_accuracy = evaluated.groupby("image_key")["is_correct"].all().mean()
    print("Overall field accuracy:", f"{evaluated['is_correct'].mean():.1%}")
    print("Complete-receipt accuracy:", f"{receipt_accuracy:.1%}")
    display(accuracy_by_split.rename("field_accuracy").to_frame())
    display(ocr_metrics.sort_values("exact_accuracy"))
else:
    evaluated = pd.DataFrame()
    ocr_metrics = pd.DataFrame()
    print("No OCR results to evaluate. Run Section 6 first.")

In [ ]:
if not ocr_metrics.empty:
    chart_data = (
        ocr_metrics[ocr_metrics["level"].ne("private")]
        .groupby("field", as_index=False)["exact_accuracy"].mean()
        .sort_values("exact_accuracy")
    )
    plt.figure(figsize=(10, 6))
    sns.barplot(data=chart_data, x="exact_accuracy", y="field", color="#1F4D3F")
    plt.xlim(0, 1)
    plt.title("OCR exact accuracy by field")
    plt.xlabel("Accuracy")
    plt.ylabel("")
    plt.tight_layout()
    plt.show()

    error_examples = evaluated[evaluated["is_correct"].eq(0)][
        ["image_key", "split", "level", "field", "actual", "predicted"]
    ]
    display(error_examples.head(20))
else:
    error_examples = pd.DataFrame()

## 8. Sales and brand analysis

Product receipts measure **store sales handled (GMV)**. They are not automatically Galleria's revenue because partner brands currently pay fixed rent.

In [ ]:
known_gmv = receipts["receipt_total_clean"].sum(min_count=1)
average_order_value = receipts["receipt_total_clean"].mean()
known_units = transactions["quantity"].sum(min_count=1)
outstanding = receipts["current_outstanding_bhd"].sum(min_count=1)

business_kpis = pd.DataFrame({
    "Metric": [
        "Unique receipts", "Known store GMV (BHD)",
        "Average order value (BHD)", "Known units/services",
        "Current outstanding balance (BHD)",
    ],
    "Value": [
        len(receipts), known_gmv, average_order_value,
        known_units, outstanding,
    ],
})
display(business_kpis)

In [ ]:
daily_sales = (
    receipts.dropna(subset=["transaction_date", "receipt_total_clean"])
    .groupby("transaction_date", as_index=False)["receipt_total_clean"].sum()
)

plt.figure(figsize=(11, 4.5))
sns.lineplot(
    data=daily_sales, x="transaction_date", y="receipt_total_clean",
    marker="o", color="#1F4D3F",
)
plt.title("Daily known store GMV")
plt.xlabel("Date")
plt.ylabel("BHD")
plt.tight_layout()
plt.show()

In [ ]:
category_summary = (
    transactions.assign(
        product_category=transactions["product_category"].fillna("Unknown")
    )
    .groupby("product_category", as_index=False)
    .agg(
        item_rows=("line_number", "size"),
        units=("quantity", "sum"),
        known_value_bhd=("line_total_clean", "sum"),
    )
    .sort_values("known_value_bhd", ascending=False)
)
display(category_summary)

plt.figure(figsize=(9, 5))
sns.barplot(data=category_summary, x="known_value_bhd", y="product_category", color="#B8904A")
plt.title("Known value by category")
plt.xlabel("BHD")
plt.ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
confirmed_brand_rows = transactions[
    ~transactions["brand_clean"].fillna("Unknown").eq("Unknown")
]
brand_coverage = len(confirmed_brand_rows) / max(len(transactions), 1)

brand_summary = confirmed_brand_rows.groupby("brand_clean", as_index=False).agg(
    item_rows=("line_number", "size"),
    units=("quantity", "sum"),
    known_value_bhd=("line_total_clean", "sum"),
    average_unit_price_bhd=("unit_price_bhd", "mean"),
).sort_values("known_value_bhd", ascending=False)

print(f"Confirmed brand coverage: {brand_coverage:.1%}")
display(brand_summary)

if brand_coverage < 0.50:
    print("Brand ranking is disabled: at least 50% confirmed coverage is required.")

In [ ]:
payment_summary = (
    receipts["final_payment_status"].value_counts()
    .rename_axis("payment_status")
    .reset_index(name="receipts")
)
transaction_type_summary = (
    transactions.groupby("transaction_type", as_index=False)
    .agg(rows=("line_number", "size"), known_value_bhd=("line_total_clean", "sum"))
    .sort_values("known_value_bhd", ascending=False)
)

display(payment_summary)
display(transaction_type_summary)

## 9. Customer groups

Customer groups are transparent business rules, not forced K-Means clusters. This is appropriate while repeat purchasing is limited.

In [ ]:
customer_features = receipts.groupby("customer_id", as_index=False).agg(
    receipts=("receipt_id", "nunique"),
    total_spend_bhd=("receipt_total_clean", lambda values: values.sum(min_count=1)),
    average_order_bhd=("receipt_total_clean", "mean"),
    last_purchase=("transaction_date", "max"),
)

alteration_customers = (
    transactions.assign(
        is_alteration=transactions["transaction_type"].eq("Alteration")
        | transactions["alteration_required"].fillna("").astype(str).str.lower().eq("yes")
    )
    .groupby("customer_id")["is_alteration"].any()
)
customer_features["alteration_customer"] = (
    customer_features["customer_id"].map(alteration_customers).fillna(False)
)

ANALYSIS_DATE = pd.Timestamp.today().normalize()
customer_features["days_since_purchase"] = (
    ANALYSIS_DATE - customer_features["last_purchase"]
).dt.days

In [ ]:
high_value_cutoff = customer_features["total_spend_bhd"].quantile(0.80)
customer_features["high_value"] = (
    customer_features["total_spend_bhd"] >= high_value_cutoff
)
customer_features["recent"] = customer_features["days_since_purchase"].le(60)
customer_features["inactive"] = customer_features["days_since_purchase"].ge(180)

def build_tags(row):
    tags = []
    if row["high_value"]:
        tags.append("High Value")
    if row["recent"]:
        tags.append("Recent")
    if row["inactive"]:
        tags.append("Inactive")
    if row["alteration_customer"]:
        tags.append("Alteration Customer")
    return ", ".join(tags) if tags else "Standard"

customer_features["audience_tags"] = customer_features.apply(build_tags, axis=1)
display(customer_features.sort_values("total_spend_bhd", ascending=False).head(15))

In [ ]:
consent_lookup = (
    customers.set_index("customer_id")["consent_for_marketing"]
    .fillna("Unknown").astype(str)
)
customer_features["marketing_consent"] = (
    customer_features["customer_id"].map(consent_lookup).fillna("Unknown")
)
customer_features["contactable"] = (
    customer_features["marketing_consent"].str.lower().eq("yes")
)

customer_group_summary = pd.DataFrame({
    "Audience": ["High Value", "Recent", "Inactive", "Alteration Customer"],
    "Customers": [
        int(customer_features["high_value"].sum()),
        int(customer_features["recent"].sum()),
        int(customer_features["inactive"].sum()),
        int(customer_features["alteration_customer"].sum()),
    ],
})
display(customer_group_summary)
print("Customers currently contactable with recorded consent:", int(customer_features["contactable"].sum()))

## 10. Campaign recommendation tool

Recommendations are explainable rules. They do not claim to predict campaign response before campaign outcomes exist.

In [ ]:
campaign_conditions = [
    customer_features["inactive"] & customer_features["high_value"],
    customer_features["inactive"],
    customer_features["high_value"],
    customer_features["alteration_customer"],
    customer_features["recent"],
]
audience_names = [
    "Inactive High Value", "Inactive", "High Value",
    "Alteration Customer", "Recent",
]
campaign_names = [
    "VIP Return Offer", "10% Return Offer", "VIP Preview",
    "Free Alteration", "Referral Reward",
]
campaign_reasons = [
    "Win back a valuable customer without a mass campaign",
    "Encourage a customer who has not purchased recently",
    "Reward strong spend without immediately discounting",
    "Use a service benefit relevant to previous behaviour",
    "Invite a recent buyer to introduce a new customer",
]

customer_features["primary_audience"] = np.select(
    campaign_conditions, audience_names, default="Standard"
)
customer_features["recommended_campaign"] = np.select(
    campaign_conditions, campaign_names, default="Collection Update"
)
customer_features["campaign_reason"] = np.select(
    campaign_conditions, campaign_reasons, default="General product awareness"
)

campaign_recommendations = customer_features.groupby(
    ["primary_audience", "recommended_campaign", "campaign_reason"],
    as_index=False,
).agg(
    eligible_customers=("customer_id", "nunique"),
    contactable_with_consent=("contactable", "sum"),
).rename(columns={"primary_audience": "audience"})
display(campaign_recommendations)

In [ ]:
CAMPAIGN_SCENARIO = {
    "audience": "Inactive",
    "campaign": "10% Return Offer",
    "expected_conversion_rate": 0.08,
    "discount_rate": 0.10,
    "contribution_margin_rate": 0.30,
    "fixed_campaign_cost_bhd": 0.0,
    "extra_cost_per_conversion_bhd": 0.0,
}

target_row = campaign_recommendations[
    campaign_recommendations["audience"].eq(CAMPAIGN_SCENARIO["audience"])
]
target_count = int(target_row["eligible_customers"].iloc[0]) if len(target_row) else 0
scenario_aov = float(average_order_value) if pd.notna(average_order_value) else 0.0
expected_conversions = target_count * CAMPAIGN_SCENARIO["expected_conversion_rate"]
expected_gmv = expected_conversions * scenario_aov

In [ ]:
discount_cost = expected_gmv * CAMPAIGN_SCENARIO["discount_rate"]
extra_cost = expected_conversions * CAMPAIGN_SCENARIO["extra_cost_per_conversion_bhd"]
contribution_before_cost = expected_gmv * CAMPAIGN_SCENARIO["contribution_margin_rate"]
estimated_net_contribution = (
    contribution_before_cost - discount_cost - extra_cost
    - CAMPAIGN_SCENARIO["fixed_campaign_cost_bhd"]
)

campaign_scenario_result = pd.DataFrame({
    "Metric": [
        "Target customers", "Expected conversions", "Expected GMV (BHD)",
        "Estimated discount cost (BHD)", "Estimated net contribution (BHD)",
    ],
    "Value": [
        target_count, expected_conversions, expected_gmv,
        discount_cost, estimated_net_contribution,
    ],
})
display(campaign_scenario_result)
print("Scenario only: replace conversion and margin assumptions with real campaign data.")

In [ ]:
campaign_tracking_template = pd.DataFrame(columns=[
    "campaign_id", "campaign_name", "customer_id", "group",
    "sent_date", "offer_code", "converted", "purchase_date",
    "revenue_bhd", "campaign_cost_bhd",
])

display(campaign_tracking_template)
print("Use Treatment and Control values in the group column to measure incremental impact.")

## 11. Weekly GMV forecasting

Forecasting is exploratory because the active history contains a 201-day gap. Aggregate only valid receipt totals to Monday-anchored weeks, do not invent zero-sales weeks for the gap, and train only on the latest continuous weekly period. Show an unavailable state if fewer than eight valid continuous weeks remain.

Compare last-week value, four-week moving average, and simple exponential smoothing using expanding-window MAE and WAPE. Produce only a next-week estimate and report the training and evaluation weeks.

In [ ]:
monthly_sales = (
    receipts.dropna(subset=["transaction_date", "receipt_total_clean"])
    .assign(month=lambda frame: frame["transaction_date"].dt.to_period("M").dt.to_timestamp())
    .groupby("month")["receipt_total_clean"].sum()
    .sort_index()
)

observed_months = len(monthly_sales)
forecast_ready = (
    DATASET_COMPLETE_FOR_FORECAST
    and observed_months >= FORECAST_MIN_MONTHS
)

forecast_readiness = pd.DataFrame({
    "Check": ["Dataset marked complete", "Observed months", "Minimum months", "Forecast ready"],
    "Value": [
        DATASET_COMPLETE_FOR_FORECAST, observed_months,
        FORECAST_MIN_MONTHS, forecast_ready,
    ],
})
display(forecast_readiness)

In [ ]:
def make_forecasts(train, horizon):
    last_value = float(train.iloc[-1])
    naive = np.repeat(last_value, horizon)
    moving_average = np.repeat(float(train.tail(3).mean()), horizon)

    alpha, beta = 0.4, 0.2
    level = float(train.iloc[0])
    trend = float(train.iloc[1] - train.iloc[0])
    for value in train.iloc[1:]:
        previous_level = level
        level = alpha * float(value) + (1 - alpha) * (level + trend)
        trend = beta * (level - previous_level) + (1 - beta) * trend
    holt = np.array([level + step * trend for step in range(1, horizon + 1)])

    return {
        "Naive": naive,
        "3-Month Moving Average": moving_average,
        "Holt Trend": holt,
    }

def forecast_scores(actual, predicted):
    error = np.asarray(actual) - np.asarray(predicted)
    mae = np.mean(np.abs(error))
    denominator = np.sum(np.abs(actual))
    wape = np.sum(np.abs(error)) / denominator if denominator else np.nan
    return mae, wape

In [ ]:
forecast_model_scores = pd.DataFrame()
sales_forecast = pd.DataFrame()

if forecast_ready:
    test_months = min(3, max(1, observed_months // 5))
    train = monthly_sales.iloc[:-test_months]
    test = monthly_sales.iloc[-test_months:]
    candidate_predictions = make_forecasts(train, test_months)

    score_rows = []
    for model_name, prediction in candidate_predictions.items():
        mae, wape = forecast_scores(test.values, prediction)
        score_rows.append({"model": model_name, "MAE": mae, "WAPE": wape})

    forecast_model_scores = pd.DataFrame(score_rows).sort_values("WAPE")
    display(forecast_model_scores)
else:
    print("Forecast not run. Complete the historical receipt upload first.")

In [ ]:
if forecast_ready:
    best_model = forecast_model_scores.iloc[0]["model"]
    future_horizon = 3
    all_predictions = make_forecasts(monthly_sales, future_horizon)
    future_values = all_predictions[best_model]
    future_months = pd.date_range(
        monthly_sales.index.max() + pd.offsets.MonthBegin(1),
        periods=future_horizon, freq="MS",
    )
    sales_forecast = pd.DataFrame({
        "month": future_months,
        "forecast_gmv_bhd": np.maximum(future_values, 0),
        "selected_model": best_model,
    })
    display(sales_forecast)

    plt.figure(figsize=(10, 4.5))
    plt.plot(monthly_sales.index, monthly_sales.values, marker="o", label="Actual")
    plt.plot(future_months, sales_forecast["forecast_gmv_bhd"], marker="o", label="Forecast")
    plt.title("Monthly GMV forecast")
    plt.ylabel("BHD")
    plt.legend()
    plt.tight_layout()
    plt.show()

## 12. Export dashboard data

Public exports exclude customer names, phone numbers, and private OCR fields. Streamlit will load these files rather than rerunning the entire notebook.

In [ ]:
def save_csv(frame, path):
    frame.to_csv(path, index=False)
    print(f"Saved {len(frame):,} rows → {path.name}")

public_transaction_columns = [
    "image_key", "receipt_id", "line_number", "transaction_date",
    "customer_id", "transaction_type", "brand_clean", "model_number",
    "description_raw", "product_category", "color", "quantity",
    "unit_price_bhd", "line_total_clean", "receipt_total_bhd",
    "payment_status", "alteration_required", "alteration_details",
]
public_receipt_columns = [
    "image_key", "receipt_id", "transaction_date", "customer_id",
    "item_rows", "units", "receipt_total_clean", "receipt_total_source",
    "amount_paid_bhd", "current_outstanding_bhd", "final_payment_status",
]

transactions_public = transactions[
    [column for column in public_transaction_columns if column in transactions]
].copy()
receipts_public = receipts[
    [column for column in public_receipt_columns if column in receipts]
].copy()

In [ ]:
customer_public_columns = [
    "customer_id", "receipts", "total_spend_bhd", "average_order_bhd",
    "last_purchase", "days_since_purchase", "high_value", "recent",
    "inactive", "alteration_customer", "audience_tags",
    "primary_audience", "recommended_campaign", "campaign_reason",
]
customer_groups_public = customer_features[customer_public_columns].copy()

contact_columns = [
    "customer_id", "customer_name", "phone_number", "consent_for_marketing"
]
private_contacts = customers[
    [column for column in contact_columns if column in customers]
].copy()
private_contacts = private_contacts[
    private_contacts["consent_for_marketing"].fillna("").str.lower().eq("yes")
]

save_csv(transactions_public, PUBLIC_DIR / "transactions_clean_public.csv")
save_csv(receipts_public, PUBLIC_DIR / "receipts_clean_public.csv")
save_csv(category_summary, PUBLIC_DIR / "category_summary.csv")
save_csv(brand_summary, PUBLIC_DIR / "brand_summary.csv")
save_csv(customer_groups_public, PUBLIC_DIR / "customer_groups_public.csv")
save_csv(campaign_recommendations, PUBLIC_DIR / "campaign_recommendations.csv")

In [ ]:
save_csv(quality_report, PUBLIC_DIR / "data_quality_report.csv")
save_csv(payment_summary, PUBLIC_DIR / "payment_summary.csv")
save_csv(campaign_tracking_template, PRIVATE_DIR / "campaign_tracking_template.csv")
save_csv(private_contacts, PRIVATE_DIR / "campaign_contacts_with_consent.csv")

if not ocr_metrics.empty:
    save_csv(ocr_metrics, PUBLIC_DIR / "ocr_metrics_by_field.csv")
    public_eval = ocr_evaluation[ocr_evaluation["level"].ne("private")]
    private_eval = ocr_evaluation[ocr_evaluation["level"].eq("private")]
    save_csv(public_eval, PUBLIC_DIR / "ocr_evaluation_public.csv")
    save_csv(private_eval, PRIVATE_DIR / "ocr_evaluation_private.csv")

if not forecast_model_scores.empty:
    save_csv(forecast_model_scores, PUBLIC_DIR / "forecast_model_scores.csv")
if not sales_forecast.empty:
    save_csv(sales_forecast, PUBLIC_DIR / "sales_forecast.csv")

In [ ]:
run_manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "workbook": WORKBOOK_PATH.name,
    "ocr_model": HF_MODEL,
    "unique_receipts": int(len(receipts)),
    "transaction_rows": int(len(transactions)),
    "confirmed_brand_coverage": float(brand_coverage),
    "successful_ocr_extractions": int(len(successful_ocr)),
    "forecast_ready": bool(forecast_ready),
    "important_limitations": [
        "Receipt count is not store footfall.",
        "Store GMV is not automatically Galleria rental revenue.",
        "Brand rankings require better confirmed brand coverage.",
        "Campaign recommendations are rules until outcomes are collected.",
        "Forecasting requires a complete multi-month transaction history.",
    ],
}

(PUBLIC_DIR / "run_manifest.json").write_text(
    json.dumps(run_manifest, indent=2), encoding="utf-8"
)
print("Dashboard exports are ready in:", OUTPUT_DIR)

## 13. Final conclusions

After running every section, your conclusion should answer four questions:

1. **OCR:** How accurate was the pretrained model on the holdout receipts, and which fields failed most often?
2. **Business:** What do the verified receipts show about sales value, products, payments, and alterations?
3. **Growth:** Which customer audiences are currently available, and which campaigns are recommended?
4. **Readiness:** Which missing fields must Galleria collect before brand reporting and forecasting become reliable?

### Current evidence-based limitations

- The labelled benchmark contains 45 receipts, not the full historical database.
- The current dates cover roughly one month, so forecasting is disabled.
- Confirmed brand labels are very limited, so definitive brand ranking is disabled.
- Marketing consent is unknown for the current customer records.
- Inventory data is still required to identify slow-moving and unsold products.

### Next step

Use the Streamlit receipt scanner to add and verify the remaining historical receipts. When the dataset is complete, rerun this notebook to update the brand dashboard, campaign audiences, and forecast automatically.